# HW01-A — Dockerized Airbnb Ops Package

Your first job is to make a tiny data package that can run the same way twice.

That sounds boring. Boring reproducibility is how production works.

By the end, you should have:

- a Python package
- a CLI command
- a Docker image
- a Docker Compose service
- a DVC stage
- a processed CSV
- a short run report

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials (for homeworks B & C, not this one). Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

Use docs when stuck. Guessing is slower.

- Python `pyproject.toml`: https://packaging.python.org/en/latest/guides/writing-pyproject-toml/
- Python CLI entry points: https://setuptools.pypa.io/en/latest/userguide/entry_point.html
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker Compose: https://docs.docker.com/compose/
- Docker volumes: https://docs.docker.com/engine/storage/volumes/
- DVC `dvc.yaml`: https://doc.dvc.org/user-guide/project-structure/dvcyaml-files
- DVC `repro`: https://doc.dvc.org/command-reference/repro

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [281]:
from pathlib import Path
import textwrap
import pandas as pd

PROJECT = Path.cwd()
for path in ['src/airbnb_ops', 'data/raw', 'data/processed', 'reports', 'tests']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_ops/__init__.py').write_text('__version__ = "0.1.0"\n')
print(PROJECT)

C:\Users\lenovo\Desktop\hw1


## 1. Raw inputs

The two cells below are given to you.

Do not change these rows. The point of HW01-A is packaging, Docker, DVC, and validation, not inventing toy data.

Files created:

```text
data/raw/listings_sample.csv
data/raw/neighbourhood_segments.csv
```

The listings file includes `host_name` and `host_id` on purpose. Your PII code must handle them.


In [282]:
# Provided starter data 1.1
# Do not edit this cell.
# Implementation note:
# - This creates the listing-level raw input.
# - The host_name and host_id columns are intentionally included.
# - Later, your PII code must drop host_name and replace host_id with a stable host_key.

listings = pd.DataFrame(
    [
        {
            "listing_id": 1,
            "neighbourhood": "Centrum-West",
            "price": 180,
            "minimum_nights": 2,
            "availability_365": 120,
            "number_of_reviews": 31,
            "host_id": 901,
            "host_name": "Alice",
        },
        {
            "listing_id": 2,
            "neighbourhood": "Centrum-West",
            "price": 210,
            "minimum_nights": 3,
            "availability_365": 80,
            "number_of_reviews": 12,
            "host_id": 902,
            "host_name": "Bob",
        },
        {
            "listing_id": 3,
            "neighbourhood": "De Pijp",
            "price": 135,
            "minimum_nights": 2,
            "availability_365": 210,
            "number_of_reviews": 44,
            "host_id": 903,
            "host_name": "Chris",
        },
        {
            "listing_id": 4,
            "neighbourhood": "Oud-Noord",
            "price": 95,
            "minimum_nights": 1,
            "availability_365": 300,
            "number_of_reviews": 9,
            "host_id": 904,
            "host_name": "Dana",
        },
        {
            "listing_id": 5,
            "neighbourhood": "Oud-Noord",
            "price": 105,
            "minimum_nights": 2,
            "availability_365": 260,
            "number_of_reviews": 18,
            "host_id": 905,
            "host_name": "Eve",
        },
        {
            "listing_id": 6,
            "neighbourhood": "De Baarsjes",
            "price": 125,
            "minimum_nights": 3,
            "availability_365": 170,
            "number_of_reviews": 27,
            "host_id": 906,
            "host_name": "Farid",
        },
        {
            "listing_id": 7,
            "neighbourhood": "De Baarsjes",
            "price": 145,
            "minimum_nights": 4,
            "availability_365": 90,
            "number_of_reviews": 21,
            "host_id": 907,
            "host_name": "Grace",
        },
        {
            "listing_id": 8,
            "neighbourhood": "Westerpark",
            "price": 155,
            "minimum_nights": 2,
            "availability_365": 140,
            "number_of_reviews": 36,
            "host_id": 908,
            "host_name": "Hamed",
        },
    ]
)

listings.to_csv(PROJECT / "data/raw/listings_sample.csv", index=False)
listings.head()


,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


In [283]:
# Provided starter data 1.2
# Do not edit this cell.
# Implementation note:
# - This file enriches neighbourhoods with business metadata.
# - Your transform step should join this file after aggregation.
# - If a neighbourhood has no segment row, your code should fill it with "unknown".

segments = pd.DataFrame(
    [
        {
            "neighbourhood": "Centrum-West",
            "tourism_segment": "tourist-heavy",
            "priority_level": "high",
        },
        {
            "neighbourhood": "De Pijp",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "Oud-Noord",
            "tourism_segment": "emerging",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "De Baarsjes",
            "tourism_segment": "local-heavy",
            "priority_level": "low",
        },
        {
            "neighbourhood": "Westerpark",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
    ]
)

segments.to_csv(PROJECT / "data/raw/neighbourhood_segments.csv", index=False)
segments


,neighbourhood,tourism_segment,priority_level
0,Centrum-West,tourist-heavy,high
1,De Pijp,mixed,medium
2,Oud-Noord,emerging,medium
3,De Baarsjes,local-heavy,low
4,Westerpark,mixed,medium


In [284]:
# Checkpoint
for file in ['data/raw/listings_sample.csv', 'data/raw/neighbourhood_segments.csv']:
    assert Path(file).exists(), f'Missing {file}'

pd.read_csv('data/raw/listings_sample.csv').head()

,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


In [285]:
listings

,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve
5,6,De Baarsjes,125,3,170,27,906,Farid
6,7,De Baarsjes,145,4,90,21,907,Grace
7,8,Westerpark,155,2,140,36,908,Hamed


## 2. Data contract

Write code against a contract, not vibes.

Input listing columns:

```text
listing_id, neighbourhood, price, minimum_nights,
availability_365, number_of_reviews, host_id, host_name
```

Output columns:

```text
neighbourhood, num_listings, avg_price, median_price,
avg_minimum_nights, availability_365_avg, total_reviews,
reviews_per_listing, tourism_segment, priority_level
```

In [286]:
# TODO 2.1
# Create src/airbnb_ops/config.py.
# Add a PipelineConfig dataclass with default paths for:
# listings_path, segments_path, output_path, report_path

# Write your code here.

In [287]:
%%writefile src/airbnb_ops/config.py

from dataclasses import dataclass

@dataclass
class PipelineConfig:
    listings_path: str
    segments_path: str
    output_path: str
    report_path: str

PipelineConfig = PipelineConfig(
    listings_path= 'data/raw/listings_sample.csv', 
    segments_path= 'data/raw/neighbourhood_segments.csv',
    output_path= 'outputs',
    report_path= 'reports'
)

Overwriting src/airbnb_ops/config.py


In [288]:
# TODO 2.2
# Create src/airbnb_ops/extract.py.
# Add read_csv_checked(path: Path) -> pd.DataFrame.
# It should raise FileNotFoundError if the file is missing.

# Write your code here.


In [289]:
%%writefile src/airbnb_ops/extract.py

import pandas as pd

def read_csv_checked(path):
    return pd.read_csv(path)


Overwriting src/airbnb_ops/extract.py


## 3. PII handling

For this homework:

- drop `host_name`
- convert `host_id` to `host_key`
- drop the original `host_id`

Do not use Python's built-in `hash()`. It is not stable across sessions. Use `hashlib.sha256`.

In [290]:
# TODO 3.1
# Create src/airbnb_ops/pii.py.
# Add:
# - DIRECT_PII_COLUMNS
# - pseudonymize_value(value, salt='qbc12')
# - handle_pii(df)

# Write your code here.

In [304]:
#%%writefile src/airbnb_ops/pii.py

import sys
from hashlib import sha256, pbkdf2_hmac

listings = read_csv_checked('data/raw/listings_sample.csv')

sha256 = hashlib.sha256()
# def pseudonymize_value(value, salt=b'qbc12'):
#     sha256.update(bytes(value)+ salt)
#     return sha256.hexdigest()
    
def pseudonymize_value(value, salt=b'qbc12'):
    dk = pbkdf2_hmac('sha256', bytes(value), salt, 1000)
    return dk.hex()

host_key = []
def handle_pii(df):
    for x in df['host_id']:
        host_key.append(pseudonymize_value(x))
    return host_key
    

listings['host_key'] = handle_pii(listings)
listings = listings.drop(columns=['host_name', 'host_id'])
listings.to_csv(PROJECT / "data/raw/listings_sample.csv", index=False)
listings


,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_key
0,1,Centrum-West,180,2,120,31,2b9dc6a8daf9f190c54c92bd980bdcfbe021717aa86f61...
1,2,Centrum-West,210,3,80,12,ac697a5d09988a0e2c580f0f42d5cb6e067f31da86ce2d...
2,3,De Pijp,135,2,210,44,7ce6baa0507d35a64f60cdae7dbc1e325e503bef56c27e...
3,4,Oud-Noord,95,1,300,9,403e65ea0bc48a4a90b85c4c459782e447465130a97360...
4,5,Oud-Noord,105,2,260,18,681d3c1505422c00c31e7b39f0857880b3ee9a6f6e9dc7...
5,6,De Baarsjes,125,3,170,27,93e52639ee7ba74fd59fe85c4e66e836f0171beab3d03d...
6,7,De Baarsjes,145,4,90,21,ca312037bd314b18a4be26fcba7cacc57a8c93eb6ee3db...
7,8,Westerpark,155,2,140,36,57908e64c20d7b01e75a987daf5f5c337f8abab9acdba8...


## 4. Transform

Build a neighbourhood-level summary. Use `groupby`, not loops.

9Join the segment metadata after aggregation. If a segment is missing, fill it with `unknown`.

In [305]:
# TODO 4.1
# Create src/airbnb_ops/transform.py.
# Add build_neighbourhood_summary(listings, segments).
# It should validate required input columns, aggregate, join segments, and return a dataframe.

# Write your code here.

In [306]:
segments = read_csv_checked('data/raw/neighbourhood_segments.csv')
listings = read_csv_checked('data/raw/listings_sample.csv')

In [376]:
gb= listings.groupby('neighbourhood')
x = gb['number_of_reviews'].groups
x

{'Centrum-West': [0, 1], 'De Baarsjes': [5, 6], 'De Pijp': [2], 'Oud-Noord': [3, 4], 'Westerpark': [7]}

In [399]:
%%writefile src/airbnb_ops/transform.py

def build_neighbourhood_summary(listings, segments):
    aggregated_data = listings.groupby(['neighbourhood'],as_index=False, group_keys=True, dropna=False).agg(
        num_listings=('listing_id', 'count'),
        avg_price=('price', 'mean'),
        median_price = ('price', 'median'),
        avg_minimum_nights = ('minimum_nights', 'mean'),
        availability_365_avg = ('availability_365', 'mean'),
        total_reviews = ('number_of_reviews', 'sum'),
        reviews_per_listing = ('number_of_reviews', 'count' ))

    return aggregated_data.join(segments.set_index("neighbourhood"), on='neighbourhood')

build_neighbourhood_summary(listings, segments)

Overwriting src/airbnb_ops/transform.py


## 5. Validation

Validation is the bouncer. Bad output does not get into the club.

Minimum checks:

- output is not empty 
- required output columns exist
- no PII columns exist
- neighbourhood is not null
- num_listings > 0
- avg_price >= 0
- availability_365_avg between 0 and 365

In [384]:
# TODO 5.1
# Create src/airbnb_ops/validate.py.
# Add validate_summary(summary) -> None.
# Raise ValueError when a check fails.

# Write your code here.

In [391]:
summary = build_neighbourhood_summary(listings, segments)

In [393]:
%%writefile src/airbnb_ops/validate.py
PII = 'host_name', 'host_id'
req_output_cols = ['neighbourhood', 'num_listings', 'avg_price', 'median_price', 'avg_minimum_nights', 'availability_365_avg', 'total_reviews',
                   'reviews_per_listing', 'tourism_segment', 'priority_level']

def validate_summary(summary):
    if summary.empty:
        return "ValueError: output is empty"
    if not summary.columns.isin(req_output_cols).any():
        return "ValueError: req_output_cols columns do not exist"
    if PII in summary.columns or PII in listings.columns:
        return "ValueError: PII columns exist"
    if summary.neighbourhood.isna().any():
        return "ValueError: neighbourhood is null"
    if not summary.query("num_listings<=0").empty:
        return "ValueError: invalid num_listings"     
    if not summary.query("avg_price<0").empty:
        return "ValueError: invalid avg_price"
    if not summary.query("availability_365_avg < 0 or availability_365_avg> 365").empty:
        return "ValueError: invalid availability_365_avg"
    


validate_summary(summary)

Overwriting src/airbnb_ops/validate.py


## 6. CLI

The package should expose one command:

```bash
airbnb-ops run
```

The command should read inputs, handle PII, transform, validate, write CSV, and write a markdown report.

In [ ]:
# TODO 6.1
# Create src/airbnb_ops/cli.py using Typer.
# Add a run command.
# It should write:
# - data/processed/airbnb_neighbourhood_summary.csv
# - reports/hw01_a_run_report.md

# Write your code here.

In [ ]:
#%%writefile src/airbnb_ops/cli.py

# Python program to take multiple
# inputs of different datatypes and print it
import typer

# Function with multiple parameters
def details(directory: str):
    extract.read_csv_checked(directory)
    pii.handle_pii()
    validate.validate_summary(summary)
  
    print(f"Country: {country}")
    if display == True:
        print("@geeksforgeeks")
    print(f"Name: {name}")
    print(f"Age: {age}")
    print(f"Marks: {marks}")


typer.run(details)

## 7. Package metadata

`pyproject.toml` makes the project installable. `[project.scripts]` creates the `airbnb-ops` command.

In [ ]:
# TODO 7.1
# Create pyproject.toml.
# Requirements:
# - project name: airbnb-ops
# - dependencies: pandas, typer, rich
# - script: airbnb-ops = airbnb_ops.cli:app

# Write your code here.

In [ ]:
# TODO 7.2
# Create requirements.txt with pandas, typer, rich, dvc.

# Write your code here.

In [ ]:
# Local smoke test
import sys
!{sys.executable} -m pip install -q -e .
!airbnb-ops run
pd.read_csv('data/processed/airbnb_neighbourhood_summary.csv').head()

## 8. Docker

Create `.dockerignore`, `Dockerfile`, and `docker-compose.yml`.

Watch out for this common garbage move: copying generated outputs into the image. Do not do that. Mount `data/` and `reports/` so the container writes outputs to your working directory.

In [ ]:
# TODO 8.1
# Create .dockerignore.
# Exclude .git, .venv, __pycache__, .ipynb_checkpoints, data/processed, reports.

# Write your code here.

In [ ]:
# TODO 8.2
# Create Dockerfile.
# Requirements:
# - FROM python:3.11-slim
# - WORKDIR /app
# - copy package metadata and src/
# - install requirements and package
# - default command: airbnb-ops run

# Write your code here.

In [ ]:
# TODO 8.3
# Create docker-compose.yml.
# Requirements:
# - service name: airbnb-ops
# - build current directory
# - mount ./data and ./reports
# - command: airbnb-ops run

# Write your code here.

In [ ]:
# Docker smoke test. Run in terminal if notebook cannot access Docker.
!docker compose build
!docker compose run --rm airbnb-ops

## 9. DVC

A DVC stage is the receipt for how your output was produced.

Create one stage named `run_pipeline`.

In [ ]:
# TODO 9.1
# Create dvc.yaml.
# Stage:
# - cmd: airbnb-ops run
# - deps: src/airbnb_ops, both raw CSV files
# - outs: processed CSV
# - metrics/report: reports/hw01_a_run_report.md with cache: false

# Write your code here.

In [ ]:
# Extra credit 9.2
# Run these commands if DVC is installed in your local environment.
# This is a useful check, but the required part is creating a correct dvc.yaml file.

# !dvc repro
# !dvc dag


## Final proof

If this cell fails, HW01-A is not done.

In [ ]:
output = Path('data/processed/airbnb_neighbourhood_summary.csv')
report = Path('reports/hw01_a_run_report.md')
assert output.exists(), 'Output file was not created.'
assert report.exists(), 'Report file was not created.'

df = pd.read_csv(output)
required = {'neighbourhood','num_listings','avg_price','median_price','avg_minimum_nights','availability_365_avg','total_reviews','reviews_per_listing','tourism_segment','priority_level'}
assert not (required - set(df.columns)), f'Missing: {required - set(df.columns)}'
for bad in ['host_name','host_id','reviewer_name','reviewer_id','listing_url','host_url']:
    assert bad not in df.columns, f'PII leaked: {bad}'
df.head()

## Commit

```bash
git add .
git commit -m "HW01-A dockerized Airbnb package"
```